[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/reto_expres/Reto_expres_4_asistente.ipynb)

# Reto exprés S12-13 · RAG: ¿responde con la política del hotel o dice «no consta»?

**Solo se tocan las celdas ✏️** (cambiar un valor y pulsar ▶). Las ⚙️ se ejecutan tal cual.

- ✏️ **1** · probar 3 umbrales y elegir **mirando solo la validación** (y decidiendo qué cuesta más: inventar o callar)
- ✏️ **2** · contadlo: titular con cifra y matriz «¿consta? / ¿responde?»
- ✏️ **3** · la aplicación: preguntar al asistente
- Al final sale una **FICHA** con todos los números para vuestra presentación.

| Qué se hace | Datos | Rival simple | Métrica |
|---|---|---|---|
| responder con el apartado correcto de la política, o decir «no consta» | `politica_alojamiento.md` + 36 preguntas (26 con respuesta, 10 sin ella) | «contestar siempre» (umbral 0) | % de decisiones correctas y coste de los errores |

### ⚙️ Preparación — pulsa ▶

In [ ]:
#@title ⚙️ Ejecuta esta celda primero
import os, re, urllib.request, time
BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"
def dato(nombre):
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta): return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEMILLA = 42

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
print("Listo ✔")

### ⚙️ Las preguntas de prueba — pulsa ▶

36 preguntas con su apartado correcto (**0** = la política no lo dice). Se reparten a la mitad: **validación** (para elegir) y **prueba** (una sola vez). Podéis añadir las vuestras al final de la lista.

In [ ]:
PREGUNTAS = [   # (pregunta, apartado de la política donde está la respuesta; 0 = no consta)
    ('¿Hasta cuándo puedo cancelar gratis una reserva flexible?', 1), ('¿Me devuelven el depósito si cancelo a tiempo?', 1),
    ('¿Qué pasa si cancelo una tarifa no reembolsable?', 1), ('¿Cuánto tiempo antes tengo que cancelar para no pagar la primera noche?', 1),
    ('¿A qué hora es el check-in?', 2), ('¿Hasta qué hora puedo dejar la habitación?', 2),
    ('¿Hay consigna para guardar el equipaje?', 2), ('¿Cuánto cuesta la salida tardía?', 2),
    ('¿Puedo llevar a mi perro?', 3), ('¿Se admiten gatos en el hotel?', 3),
    ('¿Pueden quedarse las mascotas solas en la habitación?', 3), ('¿Los niños pequeños pagan alojamiento?', 4),
    ('¿Hay descuento para niños de 10 años?', 4), ('¿Dan cuna para bebés?', 4),
    ('¿A qué hora se sirve el desayuno?', 5), ('¿Qué incluye la media pensión?', 5),
    ('¿Tienen comida sin gluten?', 5), ('¿Hay menú vegetariano?', 5),
    ('¿Aceptan pago con tarjeta?', 6), ('¿Se pueden pagar las reservas con cheque?', 6),
    ('¿Qué porcentaje cobran de depósito?', 6), ('¿Tiene wifi gratuito el alojamiento?', 7),
    ('¿Cuánto cuesta aparcar el coche?', 7), ('¿Hasta qué hora abre la piscina?', 7),
    ('¿A partir de qué hora hay que guardar silencio?', 8), ('¿Puedo fumar en la habitación?', 8),
    ('¿Tienen gimnasio?', 0), ('¿Hay servicio de lavandería?', 0),
    ('¿Se puede alquilar un coche en el hotel?', 0), ('¿Tienen spa con masajes?', 0),
    ('¿Cuántas estrellas tiene el hotel?', 0), ('¿Hay cargador para coches eléctricos?', 0),
    ('¿Se pueden celebrar bodas?', 0), ('¿Dan entradas para el Teide?', 0),
    ('¿Aceptan criptomonedas?', 0), ('¿Tienen pista de tenis?', 0),
]

### ⚙️ El asistente: trocear, indexar y recuperar — pulsa ▶

Cada apartado de la política es un **fragmento**. Para cada pregunta se busca el fragmento más parecido (TF-IDF + coseno) y se mira su **puntuación**. El **umbral** decide: si la puntuación no llega, el asistente dice «no consta».

In [ ]:
texto = open(dato("politica_alojamiento.md"), encoding="utf-8").read()
fragmentos = [re.sub(r"\s+", " ", p) for p in re.split(r"\n\n?(?=## )", texto)][1:]
PARA = ["el", "la", "los", "las", "de", "del", "en", "a", "y", "o", "un", "una", "es", "hay", "se", "que", "con", "para", "por", "puedo",
        "tienen", "tiene", "al", "mi", "su", "qué", "hora", "si", "no", "hasta", "desde"]
vec = TfidfVectorizer(strip_accents="unicode", stop_words=PARA).fit(fragmentos); M = vec.transform(fragmentos)

def recuperar(pregunta):
    s = cosine_similarity(vec.transform([pregunta]), M)[0]; i = int(s.argmax()); return i + 1, float(s[i])

df = pd.DataFrame(PREGUNTAS, columns=["pregunta", "apartado"]); df["consta"] = df.apartado > 0
rec = [recuperar(q) for q in df.pregunta]; df["encontrado"] = [r[0] for r in rec]; df["puntuacion"] = [r[1] for r in rec]
val, prueba = train_test_split(df, test_size=0.5, random_state=SEMILLA, stratify=df.consta)

def resultado(d, umbral):
    """Por pregunta: «bien», «inventa» (contesta lo que no es) o «calla» (dice no consta cuando sí consta)."""
    contesta = d.puntuacion >= umbral
    return pd.Series(np.where(~d.consta, np.where(contesta, "inventa", "bien"),
                     np.where(~contesta, "calla", np.where(d.encontrado == d.apartado, "bien", "inventa"))), index=d.index)

print(f"{len(fragmentos)} fragmentos · {len(val)} preguntas de validación · {len(prueba)} de prueba")
print("Ejemplo:", df.pregunta[0], "→ fragmento", df.encontrado[0], "con puntuación", round(df.puntuacion[0], 2))

### ✏️ 1 · Probar umbrales y elegir (solo con la validación)

Dos decisiones:
- **COSTES**: ¿cuánto duele **inventar** una respuesta (€ de un huésped mal informado) frente a **callar** (una llamada a recepción)?
- **UMBRALES** (mínimo 3 valores, entre 0 y 0,6) y **ELEGIDA**. La fila **umbral 0** es el rival: contesta siempre.

In [ ]:
COSTE_INVENTAR = 50      # ✏️ € cuando contesta mal o contesta lo que no consta
COSTE_CALLAR = 5         # ✏️ € cuando dice «no consta» y sí constaba
UMBRALES = [0.1, 0.2, 0.3]
def fila(u, nombre):
    r = resultado(val, u); inv, cal = int((r == "inventa").sum()), int((r == "calla").sum())
    return dict(candidata=nombre, umbral=u, val_bien=round(float((r == "bien").mean()), 3), inventa=inv, calla=cal, coste_eur=inv * COSTE_INVENTAR + cal * COSTE_CALLAR)
filas = [fila(0.0, "contestar siempre (rival)")] + [fila(u, f"umbral {u}") for u in UMBRALES]
tabla = pd.DataFrame(filas); display(tabla)
ELEGIDA = 0.2            # ✏️ umbral elegido, justificado con la tabla de arriba

### ⚙️ La prueba, una sola vez — pulsa ▶

Ahora sí, con las preguntas de prueba. Mirad **dónde inventa** (qué preguntas) y **dónde calla**.

In [ ]:
r = resultado(prueba, ELEGIDA); r0 = resultado(prueba, 0.0)
coste = lambda rr: int((rr == "inventa").sum()) * COSTE_INVENTAR + int((rr == "calla").sum()) * COSTE_CALLAR
bien, bien0 = float((r == "bien").mean()), float((r0 == "bien").mean())
for tipo in ("inventa", "calla"):
    for i in prueba.index[r == tipo]:
        print(f"  {tipo.upper():8} «{prueba.pregunta[i]}» (puntuación {prueba.puntuacion[i]:.2f}, fragmento {prueba.encontrado[i]}, correcto: {prueba.apartado[i] or 'no consta'})")
FICHA = dict(elegida=ELEGIDA, rival_bien=round(bien0, 3), bien=round(bien, 3), inventa=int((r == "inventa").sum()), calla=int((r == "calla").sum()),
             coste=coste(r), coste_rival=coste(r0), preguntas=len(prueba))
print(f"""
================ FICHA PARA LA PRESENTACIÓN ================
Problema ........ responder con la política o decir «no consta» · {len(df)} preguntas
Candidatas ...... {[(f['candidata'], f['val_bien'], f['coste_eur']) for f in filas]}  (decisiones correctas, coste en validación)
Elegida ......... umbral {ELEGIDA}   (costes: inventar {COSTE_INVENTAR} €, callar {COSTE_CALLAR} €)
--- PRUEBA (una sola vez; {len(prueba)} preguntas) ---
Contestar siempre ... {bien0:.0%} bien · coste {coste(r0)} €
Con umbral {ELEGIDA} ..... {bien:.0%} bien · inventa {FICHA['inventa']} · calla {FICHA['calla']} · coste {coste(r)} €
============================================================
""")

### ✏️ 2 · Contadlo (S13): titular con cifra y matriz

Cambiad **TITULAR**. La matriz cruza **lo que dice la política** con **lo que hace el asistente**.
**Cómo se cuenta (S13):** el titular es una **conclusión con cifra** (no «Resultados»), el gráfico empieza en **0** y las cifras salen de variables (f-string), no se teclean a mano. Terminad con un **límite**: ¿cuándo no os fiaríais?

In [ ]:
TITULAR = f"Con umbral {ELEGIDA} acierta {bien:.0%} y inventa {FICHA['inventa']} de {len(prueba)} respuestas; contestar siempre inventaba {int((r0 == 'inventa').sum())}"   # ✏️ vuestra conclusión
contesta = prueba.puntuacion >= ELEGIDA
m = np.array([[int((prueba.consta & contesta).sum()), int((prueba.consta & ~contesta).sum())],
              [int((~prueba.consta & contesta).sum()), int((~prueba.consta & ~contesta).sum())]])
fig, ax = plt.subplots(figsize=(5.2, 3.8)); ax.imshow(m, cmap="Blues", vmin=0)
for i in range(2):
    for j in range(2): ax.text(j, i, m[i, j], ha="center", va="center", fontsize=16, color="white" if m[i, j] > m.max() / 2 else "black")
ax.set_xticks([0, 1], ["responde", "dice «no consta»"]); ax.set_yticks([0, 1], ["la política lo dice", "la política NO lo dice"])
ax.set_xlabel("Lo que hace el asistente"); fig.suptitle(TITULAR, fontsize=9, fontweight="bold"); plt.tight_layout(); plt.show()
FICHA["titular"] = TITULAR

### ✏️ 3 · La aplicación: preguntar

Escribid una pregunta dentro de `preguntar("...")`. Probad una con palabras del documento, una con otras palabras («perro» frente a «perros») y una que la política no cubra.

In [ ]:
def preguntar(pregunta):
    i, s = recuperar(pregunta)
    print(f"«{pregunta}»  (puntuación {s:.2f})")
    print("  → " + (fragmentos[i - 1][:160] + ("…" if len(fragmentos[i - 1]) > 160 else "") if s >= ELEGIDA else "No consta en la política del alojamiento."))

preguntar("¿Se aceptan cheques?"); preguntar("escribe aquí tu propia pregunta")

### 🎤 Para la presentación (2–3 min)

Copiad la **FICHA**, la tabla de la celda ✏️ 1 y el gráfico. Una frase para cada pregunta: **¿qué umbral y por qué? · ¿qué cuesta más, inventar o callar? · ¿qué preguntas falla? · ¿lo pondríamos delante de un huésped?**